# ORCA-X — Google Colab GPU Training

This notebook runs the current canonical ORCA-X XGBoost production training pipeline using a Colab GPU. It preserves the completed ML refinements: real historical Open-Meteo data, six coastal locations, 6-hour forward target construction, point-in-time features, class weighting, temporal validation, Digha spatial holdout, and production model metadata.

**Before running:** Runtime → Change runtime type → select T4/L4 GPU.

In [ ]:
REPO_URL = 'https://github.com/Sayan260106/HackHeritage.git'
REPO_REF = 'feat/colab-gpu-ml'  # Change to main after this branch is merged.
REPO_DIR = '/content/HackHeritage'
START_DATE = '2020-01-01'
END_DATE = '2025-12-31'
LOCATIONS = ['digha_wb', 'paradip_od', 'vizag_ap', 'chennai_tn', 'goa', 'kochi_kl']
import os
os.environ['ORCA_X_DEVICE'] = 'cuda'
os.environ['ORCA_X_N_JOBS'] = '2'
print('ORCA_X_DEVICE =', os.environ['ORCA_X_DEVICE'])
print('ORCA_X_N_JOBS =', os.environ['ORCA_X_N_JOBS'])

In [ ]:
!rm -rf "$REPO_DIR"
!git clone --depth 1 --branch "$REPO_REF" "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git rev-parse --short HEAD
!git branch --show-current

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r ml/requirements-colab.txt
!nvidia-smi
import xgboost as xgb
print('XGBoost version:', xgb.__version__)
probe = xgb.XGBClassifier(n_estimators=2, max_depth=2, tree_method='hist', device='cuda', objective='multi:softprob', num_class=4)
print('Configured XGBoost device:', probe.get_params()['device'])

## Download and prepare the real historical dataset

The raw dataset does not need to be committed to Git. Colab downloads the same Open-Meteo historical weather + marine sources used by ORCA-X and rebuilds the canonical parquet.

In [ ]:
location_args = ' '.join(LOCATIONS)
!python ml/src/download_historical_marine.py --start "$START_DATE" --end "$END_DATE" --locations $location_args
!python ml/src/prepare_dataset.py

In [ ]:
import pandas as pd
df = pd.read_parquet('ml/data/processed/orca_historical_marine_risk.parquet')
print('Rows:', f'{len(df):,}')
print('Locations:', df['location_id'].nunique())
print(df['location_id'].value_counts().sort_index())
print('\nRisk distribution:')
print(df['risk_label'].value_counts().sort_index())

## Train the canonical production model on GPU

`ml/src/train.py` reads `ORCA_X_DEVICE`. With `cuda`, XGBoost uses the Colab GPU. The forward-target construction and Digha holdout remain inside the canonical training script, so the Colab run does not change the leakage controls.

In [ ]:
!python ml/src/train.py

In [ ]:
import json
from pathlib import Path
metadata = json.loads(Path('ml/models/orca_xgb_risk_metadata.json').read_text())
print('Model version:', metadata['model_version'])
print('Dataset version:', metadata['dataset_version'])
print('Prediction horizon:', metadata['prediction_horizon_hours'], 'hours')
print('Training device:', metadata['training_device'])
print('Feature count:', metadata['feature_count'])
print('Digha excluded from training:', metadata['digha_excluded_from_training'])
print('\nTemporal validation:')
print(json.dumps(metadata['evaluation']['temporal'], indent=2))
print('\nDigha spatial holdout:')
print(json.dumps(metadata['evaluation']['digha_spatial_holdout'], indent=2))

In [ ]:
import zipfile
bundle = '/content/orca_x_model_bundle.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write('ml/models/orca_xgb_risk.json', 'orca_xgb_risk.json')
    z.write('ml/models/orca_xgb_risk_metadata.json', 'orca_xgb_risk_metadata.json')
print('Created:', bundle)
print('Download it from the Colab file browser and put both files in HackHeritage/ml/models/.')

## Refinements 20–26

The later refinements are retained as leakage/provenance, point-in-time availability, robustness, reliability and uncertainty benchmarks. They are read-only evaluation/audit work and are not silently substituted for the canonical production model. The production inference contract remains point-in-time features only.